In [ ]:
import os
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from qdrant_client import QdrantClient, AsyncQdrantClient
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.openai import OpenAI
from llama_index.core import ServiceContext

llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0.5
)
load_dotenv()

# ----------------------------
# QDRANT
# ----------------------------
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
aclient = AsyncQdrantClient(url=qdrant_url, api_key=qdrant_api_key)

# ----------------------------
# EMBEDDINGS
# ----------------------------
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device="cpu"
)

# ----------------------------
# VECTOR STORE + INDEX
# ----------------------------
vector_store = QdrantVectorStore(
    client=client,
    aclient=aclient,
    collection_name="legal_BAAI_bge-m3"
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    storage_context=storage_context,
    embed_model=embed_model,
)

# ----------------------------
# PROMPT JURIDIQUE
# ----------------------------
qa_tmpl = PromptTemplate(
"""
Vous êtes un assistant juridique RAG.
"""
)

# ----------------------------
# QUERY ENGINE
# ----------------------------
query_engine = index.as_query_engine(
    similarity_top_k=40,
    rerank_top=10,
    response_mode="compact",
    text_qa_template=qa_tmpl,
    vector_store_query_mode="hybrid",

)

# ----------------------------
# TEST - EXÉCUTER UNE QUESTION
# ----------------------------
response = query_engine.query("Cite moi une directive importante dans le cadre de l'environnement au niveau européen?")
print(response.response)


Directive 2004/35/EC of the European Parliament and of the Council of 21 April 2004 on environmental liability with regard to the prevention and remedying of environmental damage.


In [12]:
for n in response.source_nodes:
    print(n.text)

of a product the use of which on a larger scale, namely the use of the product by several users, regardless of their number, results in the discharge, emission or introduction of a quantity of materials or substances, energy or ionising radiation into air, soil or water and causes or is likely to cause the death of, or serious injury to, any person or substantial damage to the quality of air, soil or water, or substantial damage to an ecosystem, animals or plants;</p>
        </cell>
      </row>
    </table>
    <table>
      <row>
        <cell>
          <p>(c)</p>
        </cell>
        <cell>
          <p>the manufacture, placing or making available on the market, export or use of substances, whether on their own, in mixtures or in articles, including their incorporation into articles, where such conduct causes or is likely to cause the death of, or serious injury to, any person, substantial damage to the quality of air, soil or water, or substantial damage to an ecosystem, anima

In [ ]:
# sans rerank Une loi importante dans le cadre de l'environnement en Europe est la Directive (UE) 2024/1203 du Parlement européen et du Conseil du 11 avril 2024 sur la protection de l'environnement par le droit pénal et remplaçant les directives 2008/99/CE et 2009/123/CE. Cette directive vise à établir des infractions pénales pour des comportements nuisibles à l'environnement, tels que la pollution de l'air, de l'eau et du sol, le transport et le traitement illégaux des déchets, ainsi que d'autres activités préjudiciables à la santé humaine et à l'environnement.
# avec rerank : Une loi importante dans le cadre de l'environnement en Europe est la Directive (UE) 2024/1203 du Parlement européen et du Conseil du 11 avril 2024 sur la protection de l'environnement par le droit pénal et remplaçant les directives 2008/99/CE et 2009/123/CE. Cette directive vise à établir des dispositions pénales pour prévenir et sanctionner les atteintes à l'environnement, notamment en criminalisant certaines actions nuisibles à l'environnement, telles que la pollution, le traitement illégal des déchets et les émissions nocives.
# compact : Une loi importante dans le cadre de l'environnement en Europe est la Directive 2008/99/EC du Parlement européen et du Conseil du 19 novembre 2008 sur la protection de l'environnement par le droit pénal. Cette directive vise à établir des mesures en matière de droit pénal pour protéger l'environnement de manière plus efficace. Elle définit des infractions environnementales, telles que le déversement, l'émission ou l'introduction de substances dans l'air, le sol ou l'eau, ainsi que la collecte, le transport, la récupération ou l'élimination des déchets. Les États membres doivent veiller à ce que de tels actes constituent une infraction pénale lorsqu'ils sont intentionnels et illégaux. Cette directive souligne l'importance de la protection de l'environnement et la nécessité de sanctions pénales pour assurer le respect des lois environnementales.

In [4]:
print(dir(response))
print(response.__dict__)
import json

def safe(obj):
    try:
        json.dumps(obj)
        return obj
    except:
        return str(obj)

clean = {k: safe(v) for k, v in response.__dict__.items()}

with open("./result_1.json", "w", encoding="utf-8") as f:
    json.dump(clean, f, ensure_ascii=False, indent=2)

['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'get_formatted_sources', 'metadata', 'response', 'source_nodes']
{'response': "La combustion de déchets en public peut entraîner des émissions de substances polluantes dans l'environnement, ce qui peut poser des risques pour la santé humaine et l'environnement. Selon la Directive 2010/75/UE du Parlement européen et du Conseil sur les émissions industrielles, il est important de prévenir ou de réduire les émissions diffuses provenant de l'incinération des déchets. Des mesures doivent être prises pour contrôler les émissions de poussières, de composés organiques et 

In [ ]:
import os
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from qdrant_client import QdrantClient, AsyncQdrantClient

from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI
from llama_index.core.memory import ChatMemoryBuffer

load_dotenv()

# -------------------------
# QDRANT
# -------------------------
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
aclient = AsyncQdrantClient(url=qdrant_url, api_key=qdrant_api_key)

# -------------------------
# MEMORY
# -------------------------
memory = ChatMemoryBuffer.from_defaults(token_limit=5000)

# -------------------------
# EMBEDDINGS
# -------------------------
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device="cpu"
)

# -------------------------
# VECTOR STORE + INDEX
# -------------------------
vector_store = QdrantVectorStore(
    client=client,
    aclient=aclient,
    collection_name="legal_BAAI_bge-m3"
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    storage_context=storage_context,
    embed_model=embed_model
)

query_engine = index.as_query_engine(similarity_top_k=30)

# -------------------------
# TOOL
# -------------------------
async def search_documents(query: str):
    response = await query_engine.aquery(query, similarity_top_k=30)

    sources = []
    for node in response.source_nodes:
        sources.append({
            "id": node.node_id,
            "text": node.text,
            "metadata": dict(node.metadata) if node.metadata else {},
            "score": float(node.score),
        })

    return {"answer": response.response, "sources": sources}

# -------------------------
# FUNCTION AGENT
# -------------------------
system_prompt = """
Vous êtes un moteur juridique. Répondez STRICTEMENT d'après les extraits fournis.
Chaque extrait inclut :
- son texte
- ses métadonnées (titre, type d'acte, numéro, date, chapitre)
Utilisez activement ces métadonnées pour éviter toute confusion.
Réponse détaillée et fidèle au texte et dans la même langue que l'utilisateur.
"""

agent = FunctionAgent(
    tools=[search_documents],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt=system_prompt,
    memory=memory
)

# -------------------------
# TEST DANS NOTEBOOK
# -------------------------
import asyncio

async def test():
    result = await agent.run("Parle-moi d’une loi importante sur l’environnement en Europe.")
    print(result.response.content)

await test()


c:\Users\alaa-\miniconda3\envs\tekno\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


JSONDecodeError: Expecting value: line 1 column 1 (char 0)